In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from torch.ao.nn.quantized.functional import threshold

# Data Loading and Visualization

In [ ]:
base_figure_path = Path.cwd() / "figures" / "nanotabpfn"
base_figure_path.mkdir(parents=True, exist_ok=True)
base_output_path = Path.cwd() / "output" / "nanotabpfn"
base_output_path.mkdir(parents=True, exist_ok=True)

In [ ]:
s1 = pd.read_pickle('../data/run_summary_s1.pickle.xz', compression='xz')
s1_1 = pd.read_pickle('../data/run_summary_s1.1.pickle.xz', compression='xz')
s1_2 = pd.read_pickle('../data/run_summary_s1.2.pickle.xz', compression='xz')


In [ ]:
s1 = pd.concat([s1, s1_1, s1_2], ignore_index=True)
s1.head(10)

In [ ]:
s1

In [ ]:
sorted(s1['parameters'].unique())

In [ ]:
# Loss smoothing

# I want to smooth the train loss so I can look at it and I want to smooth it in the intervall that I need.
# just take the val loss as reference.

In [ ]:
value_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss',
                 'val/small/val_loss']  # , 'Loss/train'
# create a figure with as many columns as value columns
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(value_columns),
    figsize=(4.5 * len(value_columns), 4),
)

for idx, row in s1.iterrows():
    for ax_idx, value_col in enumerate(value_columns):
        steps = row[value_col].index
        flops = row['total_flops'][steps].to_numpy()
        axes[ax_idx].plot(flops, row[value_col].to_numpy(), alpha=0.01, color="black")

for ax_idx, value_col in enumerate(value_columns):
    axes[ax_idx].set_title(value_col)

    axes[ax_idx].set_xscale("log")

    if "nll" in value_col.lower() or "loss" in value_col.lower():
        axes[ax_idx].set_yscale("log")
        # limit the maximum y-axis and set the min that they suggest
        axes[ax_idx].set_ylim(top=2)

save_path = base_figure_path / "learning_curves.png"
fig.savefig(save_path)

In [ ]:
quantile = 0.05
value_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/small/val_loss']  # , 'Loss/train'
top_configs = []
quantiles = {}
for value_col in value_columns:
    if "roc_auc" in value_col or "accuracy" in value_col:
        quantiles[value_col] = s1[value_col].apply(lambda x: x.iloc[-1]).quantile(1 - quantile)
    else:
        quantiles[value_col] = s1[value_col].apply(lambda x: x.iloc[-1]).quantile(quantile)

# create a figure with as many columns as value columns
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(value_columns),
    figsize=(4.5 * len(value_columns), 4),
)

for idx, row in s1.iterrows():
    for ax_idx, value_col in enumerate(value_columns):
        steps = row[value_col].index
        flops = row['total_flops'][steps].to_numpy()

        if row[value_col].iloc[-1] < quantiles[value_col] if not (
                "roc_auc" in value_col or "accuracy" in value_col) else row[value_col].iloc[-1] > quantiles[value_col]:
            axes[ax_idx].plot(flops, row[value_col].to_numpy(), alpha=0.1, color="red")
            top_configs.append(row['config_path'])
        else:
            axes[ax_idx].plot(flops, row[value_col].to_numpy(), alpha=0.01, color="black")

for ax_idx, value_col in enumerate(value_columns):
    axes[ax_idx].set_title(value_col)

    axes[ax_idx].set_xscale("log")

    if "nll" in value_col.lower() or "loss" in value_col.lower():
        axes[ax_idx].set_yscale("log")
        # limit the maximum y-axis and set the min that they suggest
        axes[ax_idx].set_ylim(top=2)

# set figure title
fig.suptitle(f"Learning Curves - Top {int(quantile*100)}% Highlighted in Red")
save_path = base_figure_path / "learning_curves_top.png"
fig.savefig(save_path)

top_configs = sorted(list(set(top_configs)))
with open(base_output_path / "top_configs.txt", "w") as f:
    for config in top_configs:
        f.write(config + "\n")

In [ ]:
# Todo: Smooth the training loss

In [ ]:
config_columns = [col for col in s1.columns if col.startswith('config/')]
# drop constant columns
constant_columns = [col for col in config_columns if s1[col].nunique() == 1]
config_columns = [col for col in config_columns if col not in constant_columns]
print("config columns: ", config_columns)
loss_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']
every_step_columns = ['total_flops', 'total_cells']
df = None
for loss_column in loss_columns:
    next_df = pd.DataFrame(pd.concat(s1[loss_column].to_list(), keys=s1.index))
    if df is None:
        df = next_df
    else:
        df = df.join(next_df, how='outer', validate='one_to_one')

for every_step_column in every_step_columns:
    next_df = pd.DataFrame(pd.concat(s1[every_step_column].to_list(), keys=s1.index))
    df = df.join(next_df, how='left', validate='one_to_one')

df = df.join(s1[config_columns + ["parameters", "config_path"]], how='left', on=df.index.get_level_values(0)).drop(
    columns="key_0")


In [ ]:
df.head()

# Scaling Trends

In [ ]:
def get_pareto_frontier(
        df: pd.DataFrame,
        x_name="flops",
        y_name="Validation Loss",
        minimization=True,
) -> pd.DataFrame:
    """ Function to compute Pareto over FLOPs.
    """
    # NOTE: strict assumption here that x_name is maximized and y_name is minimized
    df_copy = df.copy()
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    df_sorted = df_copy.sort_values(by=[x_name, y_name], ascending=[True, True])
    df_sorted = df_sorted.drop_duplicates(subset=[x_name], keep="first")
    pareto_points = []
    min_loss_so_far = float('inf')
    for _, row in df_sorted.iterrows():
        if row[y_name] < min_loss_so_far:
            pareto_points.append(row)
            min_loss_so_far = row[y_name]

    df_copy = pd.DataFrame(pareto_points)
    if not minimization:
        df_copy[y_name] = - df_copy[y_name]
    return df_copy


pareto = get_pareto_frontier(df, 'total_flops', 'real_data/nll')
pareto

In [ ]:
import math
import numpy as np

values = np.array([
    1.00000000e+11, 1.77827941e+11, 3.16227766e+11, 5.62341325e+11,
    1.00000000e+12, 1.77827941e+12, 3.16227766e+12, 5.62341325e+12,
    1.00000000e+13, 1.77827941e+13, 3.16227766e+13, 5.62341325e+13,
    1.00000000e+14, 1.77827941e+14, 3.16227766e+14, 5.62341325e+14,
    1.00000000e+15, 1.77827941e+15, 3.16227766e+15, 5.62341325e+15,
    1.00000000e+16, 1.77827941e+16, 3.16227766e+16, 5.62341325e+16,
    1.00000000e+17
])


def substitute_closest(x, arr=values):
    idx = np.abs(arr - x).argmin()
    return arr[idx]


def round_sig(x, sig=2):
    if x == 0:
        return 0
    return round(x, sig - int(math.floor(math.log10(abs(x)))) - 1)


# df['total_flops_rounded'] = df['total_flops'].map(lambda x: round_sig(x, 2))
df['total_flops_rounded'] = df['total_flops'].map(substitute_closest)

# Add additional columns to analyse
df['prior_max_cells'] = df['config/max_features'] * df['config/num_datapoints_max']
df['embedding_layer_ratio'] = df['config/model_config.embedding_size'] / df['config/model_config.num_layers']
df['prior_datapoint_feature_ratio'] = df['config/num_datapoints_max'] / df['config/max_features']

flops_column = 'total_flops_rounded'

trend_columns = config_columns + ["parameters", "total_cells"] + ["prior_max_cells", "embedding_layer_ratio",
                                                                  "prior_datapoint_feature_ratio"]
loss_columns = ['real_data/nll', 'real_data/roc_auc', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss']

fig, axes = plt.subplots(
    nrows=len(trend_columns) + 1,
    ncols=len(loss_columns),
    figsize=(4.5 * len(loss_columns), 3.5 * (len(trend_columns) + 1)),
    sharex=True,
    layout='constrained',
)

for loss_idx, loss_column in enumerate(loss_columns):

    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization
    pareto = get_pareto_frontier(df, flops_column, loss_column, minimization=minimization)
    for trend_idx, trend_column in enumerate([loss_column] + trend_columns):
        ax = axes[trend_idx, loss_idx]
        if loss_idx == 0:
            ax.set_ylabel(trend_column.split('/')[-1])
        if trend_idx == 0:
            ax.set_title(loss_column)
        ax.plot(pareto[flops_column], pareto[trend_column], marker='x')
        ax.set_xscale("log")

        # Set the y-axis
        possible_y_values = df[trend_column].unique()

        if "weight_decay" in trend_column:
            ax.set_yscale("symlog", linthresh=min(possible_y_values[possible_y_values > 0]))
            ax.set_ylim(-min(possible_y_values[possible_y_values > 0]) * .2, max(possible_y_values) * 1.2)
        elif trend_idx == 0 and minimization:  # adding the respective loss
            ax.set_yscale("log")
        else:
            ax.set_yscale("log")
            ax.set_ylim(min(possible_y_values) * 0.8, max(possible_y_values) * 1.2)

        if len(possible_y_values) < 20:
            # maybe indicate it with hlines
            for y in possible_y_values:
                ax.axhline(y=y, color='lightgray', linewidth=1)

fig.supxlabel("FLOPs")
# save the figure
save_path = base_figure_path / "nanotabpfn_trends.png"
fig.savefig(save_path)

## Impact of multiple seeds on Dominated Fronts 

In [ ]:
fronts = 5
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(loss_columns),
    figsize=(4.5 * len(loss_columns), 5),
    sharex=True,
    layout='constrained',
)
# gradient colors
colors = plt.cm.Greens_r(np.linspace(0, 0.7, fronts))
pareto_summary = pd.DataFrame()

for loss_idx, loss_column in enumerate(loss_columns):
    ax = axes[loss_idx]
    df_front = df.copy()
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization
    for front in range(fronts):
        pareto = get_pareto_frontier(df_front, flops_column, loss_column, minimization=minimization)
        pareto_summary = pd.concat([pareto_summary,
                                    pareto[["config_path", flops_column, loss_column]].assign(front=front + 1,
                                                                                              metric=loss_column)],
                                   ignore_index=True)
        df_front = df_front[~df_front.index.isin(pareto.index)]

        plotting_arguments = {
            "marker": 'x',
            "color": colors[front],
            "where": "post"
        }
        if loss_idx == 0:
            plotting_arguments["label"] = "Front " + str(front + 1)
        ax.step(pareto[flops_column], pareto[loss_column], **plotting_arguments)
        # ax.plot(pareto[flops_column], pareto[loss_column], color=colors[front], alpha=0.25)

    ax.set_xscale("log")
    ax.set_ylabel(loss_column)

    if "nll" in loss_column.lower() or "loss" in loss_column.lower():
        ax.set_yscale("log")

fig.supxlabel("FLOPs")
# legend at top

fig.legend(loc='outside upper center', ncol=fronts, borderaxespad=0.1)
save_path = base_figure_path / "pareto_fronts.png"
fig.savefig(save_path)

save_path = base_output_path / "pareto_summary.csv"
pareto_summary.to_csv(save_path, index=False)

In [ ]:
paths_per_metric = pareto_summary.value_counts(subset=['metric', 'config_path']).reset_index()
print("Total configurations for all paretofronts: ", len(paths_per_metric['config_path'].unique()))
print("Total configurations summed over all metrics", len(paths_per_metric))

# Todo: Calculate the hypervolume

In [ ]:
import re
base_paths = paths_per_metric['config_path'].unique()

# substitude s1 s1.1 and s1.2 with s1_seed=1 s1.1_seed=1 s1.2_seed=1 respectively
seed_paths = []

def replace_seed(path, seed):
    return re.sub(r'(scaling/)(s1(?:\.\d)?)', rf'\1\2_seed={seed}', path)


for seed in ['1826', '1926', '2126', '2226']:
    seed_path = [replace_seed(p, seed) for p in base_paths]
    seed_paths.extend(seed_path)

with open(base_output_path / "eval_seed_paths.txt", "w") as f:
    for path in seed_paths:
        f.write(path + "\n")

## Analyse the top configurations

### Distribution of Top Configurations across FLOPs

### Interactive Parallel Cooridinate Plots
The plots are saved as HTML and not displayed here in the notebook since it made my Computer slow 

In [ ]:
import plotly.graph_objects as go


def create_parallel_coordinates_arguments(df, config_columns, loss_column, is_min=True, quantile=0.01):
    if is_min:
        top_quantile_threshold = df[loss_column].quantile(quantile)
        subset_df = df[df[loss_column] <= top_quantile_threshold].copy()
    else:
        top_quantile_threshold = df[loss_column].quantile(1 - quantile)
        subset_df = df[df[loss_column] >= top_quantile_threshold].copy()

    dimensions = []
    for col in config_columns + [loss_column]:
        dim = {}

        dim["label"] = col.split('/')[-1].split('.')[-1]
        if loss_column == col:
            dim["values"] = subset_df[col]
            # min_val = dim["values"].min()
            # max_val = dim["values"].max()
            # dim["tickvals"]=list(np.linspace(min_val, max_val, 5)),
            # dim["ticktext"]=[f"{v:.3f}" for v in np.linspace(min_val, max_val, 5)]
            # dim["range"] = [min_val, max_val]   
        else:
            unique_values = sorted(df[col].unique())
            if "weight_decay" in col:
                dim["tickvals"] = list(range(len(unique_values)))
                dim["values"] = subset_df[col].map(lambda x: unique_values.index(x))
            else:
                dim["tickvals"] = list(np.log(unique_values))
                dim["values"] = np.log(subset_df[col])

            dim["ticktext"] = [str(val) for val in unique_values]
            dim["range"] = [min(dim["tickvals"]), max(dim["tickvals"])]

        dimensions.append(dim)

    line_color = subset_df[loss_column]
    return dimensions, line_color


flops_list = sorted(df['total_flops_rounded'].unique())

for loss_column in ['real_data/nll', 'val/val_loss', 'val/small/val_loss', 'val/medium/val_loss', 'real_data/roc_auc']:
    fig = go.Figure()
    subset_df = df[df['total_flops_rounded'] == flops_list[0]]
    is_min = not ("roc_auc" in loss_column or "accuracy" in loss_column)
    dimensions, line_color = create_parallel_coordinates_arguments(subset_df, config_columns, loss_column,
                                                                   quantile=0.01, is_min=is_min)
    fig.add_trace(
        go.Parcoords(
            line=dict(color=line_color, colorscale='Viridis', showscale=True),
            dimensions=dimensions
        )
    )

    buttons = []

    for flops in flops_list:
        subset_df = df[df['total_flops_rounded'] == flops]
        dimensions, line_color = create_parallel_coordinates_arguments(subset_df, config_columns, loss_column,
                                                                       quantile=0.01)
        buttons.append(dict(
            label=f"{flops:3.2e} FLOPs",
            method="restyle",
            # What changes when dropdown is clicked
            args=[{
                "dimensions": dimensions,
                "line.color": line_color
            }]
        ))

    fig.update_layout(
        updatemenus=[dict(
            buttons=buttons,
            direction="down",
            x=1.02,  # slightly outside plot
            y=1.25,  # above plot
            xanchor="left",
            yanchor="top",
            showactive=True,
            bgcolor="white",
        )],
        title=loss_column + " - Top 1% Configurations"
    )
    # Save interactive version
    save_path = base_figure_path / "parallel_coord" / f"{loss_column}.html".replace("/", "_")
    # create if it does not exis
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.write_html(save_path, include_plotlyjs="full")

In [ ]:
loss_columns

## Hyperparameter Importance

### Pareto Front Comparisons

In [ ]:
fig, axes = plt.subplots(
    nrows=1,
    ncols=len(loss_columns),
    figsize=(4.5 * len(loss_columns), 5),
    sharex=True,
    layout='constrained',
)
# gradient colors
settings = [
    {
        "label": "All Configurations",
        "filters": {
        }},
    {
        "label": "wd=0",
        "filters": {
            "config/weight_decay": 0.,
        }},
    {
        "label": "wd=0, bs=64",
        "filters": {
            "config/weight_decay": 0.,
            "config/effective_batch_size": 64,
        }},
    {
        "label": "bs=64",
        "filters": {
            "config/effective_batch_size": 64,
        }}
]

for loss_idx, loss_column in enumerate(loss_columns):
    ax = axes[loss_idx]
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    minimization = not maximization
    for setting in settings:
        setting_df = df
        for filter_col, filter_val in setting["filters"].items():
            setting_df = setting_df[setting_df[filter_col] == filter_val]
        pareto = get_pareto_frontier(setting_df, flops_column, loss_column, minimization=minimization)
        ax.plot(pareto[flops_column], pareto[loss_column], label=setting["label"] if loss_idx == 0 else '', marker='x')

    ax.set_xscale("log")
    ax.set_ylabel(loss_column)

    if "nll" in loss_column.lower() or "loss" in loss_column.lower():
        ax.set_yscale("log")

fig.supxlabel("FLOPs")
fig.legend(loc='outside upper center', ncol=len(settings), borderaxespad=0.1)
save_path = base_figure_path / "pareto_hyperparameter_comparison.png"
fig.savefig(save_path)

#### Heatmap Loss

In [ ]:
import seaborn as sns
fig, axes = plt.subplots(
    nrows=len(config_columns),
    ncols=len(loss_columns),
    figsize=(7.5 * len(loss_columns), 5.5 * (len(config_columns))),
    layout='constrained',
)

for loss_idx, loss_column in enumerate(loss_columns):
    # For all flop computes take the top quantile
    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    # I need this shit for each flop level...

    for trend_idx, trend_column in enumerate(config_columns):
        ax = axes[trend_idx, loss_idx]
        best = pd.pivot_table(
            df,
            values=loss_column, # does cot matter
            index=flops_column,
            columns=trend_column,
            aggfunc="max" if maximization else "min"
        )
        best.index = best.index.map(lambda x: f"{x:.1e}")
        sns.heatmap(best, annot=True, fmt=".2f", cmap="Greens", ax=ax)
        if trend_idx == 0:
            ax.set_title(loss_column)

# save figure
save_path = base_figure_path / "heatmap_best_config_loss.png"
fig.savefig(save_path)

#### Discrete Ridgeline

In [ ]:
def plot_ridgeline(ax, data):
    spacing = 1.01
    
    categories = list(data.columns)
    x = np.arange(len(categories))
    # get continous colorbar values
    
    colors = plt.cm.Blues_r(np.linspace(0.25, 0.7, len(data)))
    
    for i, (idx, row) in enumerate(data.iterrows()):
        baseline = i * spacing
        ax.bar(
            x,
            row.to_numpy() / row.sum(),
            width=1.0,
            align='edge',
            bottom=baseline,
            edgecolor='grey',
            linewidth=0.8,
            color=colors[i]
        )
        ax.axhline(y=baseline, linewidth=0.8, color='grey')
    
    ax.set_yticks([i * spacing for i in range(len(data))])
    ax.set_yticklabels([f"{idx:.1e}" for idx in data.index])
    
    
    ax.set_xticks(x + 0.5)  # center labels on the bars (align='edge' with width=1)
    ax.set_xticklabels([str(c) for c in categories], rotation=45, ha="right")

    
    ax.spines[['top', 'right', 'left']].set_visible(False)
    ax.tick_params(left=False)

quantile = 0.05
fig, axes = plt.subplots(
    nrows=len(config_columns),
    ncols=len(loss_columns),
    figsize=(4.5 * len(loss_columns), 7.5 * (len(config_columns))),
    layout='constrained',
)

for loss_idx, loss_column in enumerate(loss_columns):
    # For all flop computes take the top quantile

    maximization = "roc_auc" in loss_column or "accuracy" in loss_column
    # I need this shit for each flop level...
    flop_quantile_thresholds = {}
    for flop_level in df[flops_column].unique():
        flop_df = df[df[flops_column] == flop_level]
        if not maximization:
            top_quantile_threshold = flop_df[loss_column].quantile(quantile)
        else:
            top_quantile_threshold = flop_df[loss_column].quantile(1 - quantile)
        flop_quantile_thresholds[flop_level] = top_quantile_threshold
    
    if maximization:
        best_config_df = df[df[loss_column] >= df[flops_column].map(lambda x: flop_quantile_thresholds[x])]
    else:
        best_config_df = df[df[loss_column] <= df[flops_column].map(lambda x: flop_quantile_thresholds[x])]


    for trend_idx, trend_column in enumerate(config_columns):
        ax = axes[trend_idx, loss_idx]
        counts = pd.pivot_table(
            best_config_df,
            values=loss_column, # does cot matter
            index=flops_column,
            columns=trend_column,
            aggfunc="count"
        ).sort_index(ascending=False)
        plot_ridgeline(ax, counts)
        ax.set_title(loss_column + " - " + trend_column)
        ax.set_xlabel(trend_column.split('/')[-1].split('.')[-1])
        ax.set_ylabel(flops_column)
        

fig.suptitle("Config Distribution among Top " + str(int(quantile*100)) + "% Configurations per FLOP Level")
save_path = base_figure_path / "ridgeline_top.png"
fig.savefig(save_path)

### Shapley Values

In [ ]:
import shap
import numpy as np
from sklearn.ensemble import RandomForestRegressor


def plot_hyperparameter_importance(df, config_columns, metrics, flops):
    X = df[config_columns].copy()
    for col in config_columns:
        # Discretize the values...
        # X[col] = 
        pass

    # fit a model

    for metric in metrics:
        fig, ax = plt.subplots(figsize=(10, 5))

        y = df[metric]
        # model = TabPFNRegressor()
        model = RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)
        model.fit(X, y)

        explainer = shap.Explainer(model)
        shap_values = explainer(X)
        shap.plots.beeswarm(shap_values, show=False, ax=ax)
        fig.title("Hyperparameter Importance - " + metric + " - FLOPs: " + str(flops))
        plt.show()
        save_path = base_figure_path / "hyperparameter_importance" / (metric + "_flops_" + str(flops) + ".png")
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path)
        # spearman correlation

    corr = df[metrics].corr(method='spearman')

    return corr


for flops in sorted(df['total_flops_rounded'].unique()):
    print("FLOPs: ", flops)
    subset_df = df[df['total_flops_rounded'] == flops]
    plot_hyperparameter_importance(subset_df, config_columns, loss_columns, flops)

In [ ]:
# I should look how I plotted hyperparameter importance before...
# Search the cluster for that notebook
# Todo: Changes - Use TabPFN for the HP imoprtance --- Hoptentially look at HyperSHAP

plot_hyperparameter_importance(df, config_columns, loss_columns)

In [ ]:
# Parallel Coordinate PLots of top 10% Hyperparameter Configurations for each loss metric

# 

In [ ]:

# I want hyperparameter importance based on tabPFN for all the hyperparmeters

# For reruns: Higher learning rate, Lower hidden dimension, more layers?

In [ ]:
sorted(df['total_flops_rounded'].unique())

In [ ]:
df['total_flops_rounded'].value_counts()